# ACTIVE-TB-S: Full 156-Task Spec-Only Oracle Generation

This phase receives only `task_id`, natural-language `description`, and module `header`. It generates independent Qwen and Llama RTL behavioral-oracle candidates. Golden RTL, official tests, and benchmark mutants are physically absent. A resumable ZIP is atomically refreshed every eight generated candidates.


In [ ]:
from pathlib import Path
import json, os, shutil, subprocess, sys, zipfile
os.environ['PYTHONDONTWRITEBYTECODE']='1'
os.environ['TOKENIZERS_PARALLELISM']='false'
os.environ['HF_HOME']='/kaggle/working/hf_cache'
os.environ['TRANSFORMERS_CACHE']='/kaggle/working/hf_cache'
INPUT=Path('/kaggle/input'); WORK=Path('/kaggle/working')
markers=list(INPUT.rglob('active_tb_s_oracle_full_v1/LEAKAGE_BOUNDARY.json'))
if len(markers)==1: SOURCE=markers[0].parent
else:
    archives=list(INPUT.rglob('active_tb_s_oracle_full_v1.zip')); assert len(archives)==1, archives
    attached=WORK/'attached_full_oracle_source'
    with zipfile.ZipFile(archives[0]) as zf: zf.extractall(attached)
    SOURCE=attached/'active_tb_s_oracle_full_v1'
PROJECT=WORK/'active_tb_s_oracle_full_project'
if PROJECT.exists(): shutil.rmtree(PROJECT)
shutil.copytree(SOURCE/'project',PROJECT)
SPEC_TASKS=SOURCE/'spec_tasks_full.jsonl'
OUT=WORK/'active_tb_s_oracle_full_candidates_v1'; OUT.mkdir(exist_ok=True)
SNAPSHOT=WORK/'active_tb_s_oracle_full_checkpoint.zip'

# Optional resume input: attach a prior checkpoint dataset and rerun unchanged.
resume_archives=[p for p in INPUT.rglob('active_tb_s_oracle_full_checkpoint.zip')]
if len(resume_archives)>1: raise RuntimeError(f'Multiple resume checkpoints: {resume_archives}')
if resume_archives:
    resume_dir=WORK/'resume_full_oracle'
    with zipfile.ZipFile(resume_archives[0]) as zf: zf.extractall(resume_dir)
    for name in ['qwen_candidates.jsonl','llama_candidates.jsonl']:
        matches=list(resume_dir.rglob(name))
        if len(matches)==1: shutil.copy2(matches[0],OUT/name)
print('Sanitized source:',SOURCE,'resume:',resume_archives)


In [ ]:
boundary=json.loads((SOURCE/'LEAKAGE_BOUNDARY.json').read_text())
spec_rows=[json.loads(line) for line in SPEC_TASKS.read_text().splitlines() if line.strip()]
assert len(spec_rows)==156
assert all(set(row)=={'task_id','description','header'} for row in spec_rows)
assert boundary['golden_artifacts_in_dataset'] is False
assert not list(SOURCE.rglob('*mutant*'))
print('FULL_SPEC_ONLY_LEAKAGE_AUDIT_PASS:',len(spec_rows),'tasks')


In [ ]:
subprocess.run(['apt-get','update','-qq'],check=True)
subprocess.run(['apt-get','install','-y','-qq','iverilog'],check=True)
subprocess.run([sys.executable,'-m','pip','install','-q','--no-cache-dir','--force-reinstall',
                'torch==2.5.1','--index-url','https://download.pytorch.org/whl/cu121'],check=True)
subprocess.run([sys.executable,'-m','pip','uninstall','-y','-q','torchvision','torchaudio'],check=False)
subprocess.run([sys.executable,'-m','pip','install','-q','--no-cache-dir',
                'transformers==4.57.6','accelerate>=1.0,<2','bitsandbytes==0.48.2','tqdm>=4.66'],check=True)
probe=subprocess.run([sys.executable,'-c',
 'import torch; print(torch.__version__,torch.cuda.get_device_name(0),torch.cuda.get_device_capability(0)); x=torch.ones(1,device="cuda"); print("CUDA_PROBE_PASS",x.item())'],
 text=True,capture_output=True)
print(probe.stdout)
if probe.returncode: raise RuntimeError(probe.stderr)
subprocess.run([sys.executable,str(PROJECT/'spec_only_smoke_test.py')],check=True,cwd=PROJECT)


In [ ]:
QWEN_MODEL='unsloth/Qwen2.5-Coder-7B-Instruct-bnb-4bit'
QWEN_OUT=OUT/'qwen_candidates.jsonl'
cmd=[sys.executable,str(PROJECT/'generate_spec_oracles.py'),
 '--tasks',str(SPEC_TASKS),'--output',str(QWEN_OUT),
 '--model',QWEN_MODEL,'--family','qwen','--candidates-per-task','2',
 '--max-new-tokens','900','--temperature','0.20','--repair-attempts','1','--resume',
 '--snapshot-path',str(SNAPSHOT),'--snapshot-root',str(OUT),'--snapshot-every','8']
subprocess.run(cmd,check=True,cwd=PROJECT)
print('Qwen rows:',len(QWEN_OUT.read_text().splitlines()))


In [ ]:
import gc
gc.collect()
try:
 import torch
 torch.cuda.empty_cache()
except Exception: pass
cache=Path(os.environ['HF_HOME'])
if cache.exists(): shutil.rmtree(cache)
cache.mkdir(parents=True,exist_ok=True)
print('Qwen cache released; checkpoint bytes:',SNAPSHOT.stat().st_size)


In [ ]:
LLAMA_MODEL='unsloth/Llama-3.1-8B-Instruct-unsloth-bnb-4bit'
LLAMA_OUT=OUT/'llama_candidates.jsonl'
cmd=[sys.executable,str(PROJECT/'generate_spec_oracles.py'),
 '--tasks',str(SPEC_TASKS),'--output',str(LLAMA_OUT),
 '--model',LLAMA_MODEL,'--family','llama','--candidates-per-task','3',
 '--max-new-tokens','900','--temperature','0.20','--repair-attempts','1','--resume',
 '--snapshot-path',str(SNAPSHOT),'--snapshot-root',str(OUT),'--snapshot-every','8']
subprocess.run(cmd,check=True,cwd=PROJECT)
print('Llama rows:',len(LLAMA_OUT.read_text().splitlines()))


In [ ]:
expected={QWEN_OUT:312,LLAMA_OUT:468}
for path,count in expected.items():
 rows=[json.loads(line) for line in path.read_text().splitlines() if line.strip()]
 assert len(rows)==count,(path,len(rows),count)
 assert len({(row['task_id'],row['candidate_index']) for row in rows})==count
 print(path.name,'compiled',sum(bool(row['compiled']) for row in rows),'/',count)
manifest={'protocol':'full spec-only oracle generation','task_count':156,
 'input_fields':['task_id','description','header'],'models':[QWEN_MODEL,LLAMA_MODEL],
 'golden_artifacts_seen':False}
(OUT/'oracle_manifest.json').write_text(json.dumps(manifest,indent=2))
if cache.exists(): shutil.rmtree(cache)
archive=WORK/'active_tb_s_oracle_full_candidates_v1.zip'
shutil.make_archive(str(archive.with_suffix('')),'zip',OUT)
print('Final checkpoint:',archive,archive.stat().st_size,'bytes')
